# LSTM Melody Generation

**CS 89.02 / MUS 14.05 — Music and AI — Week 5**

In Weeks 3 and 4, we generated melodies using Markov chains — a model that predicts
the next note based on a fixed window of recent notes. Markov models are simple and
effective for short-range patterns, but they have no mechanism for remembering what
happened 10 or 100 notes ago.

**Recurrent Neural Networks (RNNs)** address this by maintaining a hidden state that
is updated at each time step. In principle, this hidden state can carry information
across arbitrarily long sequences. In practice, vanilla RNNs suffer from the
**vanishing gradient problem**: during training, gradients shrink exponentially as
they are backpropagated through many time steps, making it hard to learn long-range
dependencies.

**Long Short-Term Memory (LSTM)** networks solve this with a gated architecture:
- **Forget gate**: decides what to discard from the cell state
- **Input gate**: decides what new information to store
- **Output gate**: decides what to output from the cell state

These gates allow the LSTM to selectively remember and forget, making it much better
at capturing patterns across longer sequences — like repeating melodic motifs,
phrase structure, or tonal centers.

In this notebook, we will:
1. Prepare a small dataset of monophonic melodies
2. Build a 2-layer LSTM in PyTorch
3. Train it to predict the next note
4. Generate melodies at different temperatures
5. Compare the results to Markov chain output

In [ ]:
!pip install -q pretty_midi matplotlib numpy torch

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pretty_midi
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Training Data

We need a corpus of monophonic melodies represented as sequences of MIDI note
numbers. For this demo, we will encode several well-known melodies by hand.
Each melody is a list of MIDI pitch values (60 = middle C).

We also include a special **REST** token (value 0) and treat all notes as having
equal duration for simplicity. The model learns to predict pitch sequences only.

In a production system, you would load thousands of MIDI files and extract
monophonic lines, but this small dataset is enough to demonstrate the architecture.

In [ ]:
# Define training melodies as MIDI note sequences
# We use a range of well-known tunes to give the model diverse patterns

REST = 0

melodies = {
    # Twinkle Twinkle Little Star (C major)
    "twinkle": [60, 60, 67, 67, 69, 69, 67, REST,
                65, 65, 64, 64, 62, 62, 60, REST,
                67, 67, 65, 65, 64, 64, 62, REST,
                67, 67, 65, 65, 64, 64, 62, REST,
                60, 60, 67, 67, 69, 69, 67, REST,
                65, 65, 64, 64, 62, 62, 60, REST],

    # Ode to Joy (D major, transposed to C)
    "ode_to_joy": [64, 64, 65, 67, 67, 65, 64, 62,
                   60, 60, 62, 64, 64, 62, 62, REST,
                   64, 64, 65, 67, 67, 65, 64, 62,
                   60, 60, 62, 64, 62, 60, 60, REST],

    # Frere Jacques
    "frere_jacques": [60, 62, 64, 60, 60, 62, 64, 60,
                      64, 65, 67, REST, 64, 65, 67, REST,
                      67, 69, 67, 65, 64, 60, 67, 69, 67, 65, 64, 60,
                      60, 55, 60, REST, 60, 55, 60, REST],

    # Mary Had a Little Lamb
    "mary": [64, 62, 60, 62, 64, 64, 64, REST,
             62, 62, 62, REST, 64, 67, 67, REST,
             64, 62, 60, 62, 64, 64, 64, 64,
             62, 62, 64, 62, 60, REST, REST, REST],

    # Happy Birthday (simplified, C major)
    "happy_birthday": [60, 60, 62, 60, 65, 64, REST,
                       60, 60, 62, 60, 67, 65, REST,
                       60, 60, 72, 69, 65, 64, 62, REST,
                       70, 70, 69, 65, 67, 65, REST, REST],

    # Au Clair de la Lune
    "au_clair": [60, 60, 60, 62, 64, REST, 62, REST,
                 60, 64, 62, 62, 60, REST, REST, REST,
                 60, 60, 60, 62, 64, REST, 62, REST,
                 60, 64, 62, 62, 60, REST, REST, REST],

    # Simple ascending/descending scale patterns
    "scale_up_down": [60, 62, 64, 65, 67, 69, 71, 72,
                      72, 71, 69, 67, 65, 64, 62, 60,
                      60, 62, 64, 65, 67, 69, 71, 72,
                      72, 71, 69, 67, 65, 64, 62, 60],

    # Arpeggiated patterns
    "arpeggios": [60, 64, 67, 72, 67, 64, 60, REST,
                  62, 65, 69, 74, 69, 65, 62, REST,
                  64, 67, 71, 76, 71, 67, 64, REST,
                  65, 69, 72, 77, 72, 69, 65, REST],
}

# Repeat each melody several times for more training data
all_notes = []
for name, melody in melodies.items():
    for _ in range(10):  # repeat each melody 10 times
        all_notes.extend(melody)

print(f"Total training tokens: {len(all_notes)}")
print(f"Unique pitches: {sorted(set(all_notes))}")
print(f"Vocabulary size: {len(set(all_notes))}")

In [ ]:
# Build vocabulary: map MIDI notes to contiguous indices
vocab = sorted(set(all_notes))
note_to_idx = {note: idx for idx, note in enumerate(vocab)}
idx_to_note = {idx: note for note, idx in note_to_idx.items()}
vocab_size = len(vocab)

print(f"Vocabulary size: {vocab_size}")
print(f"Note-to-index mapping: {note_to_idx}")

# Create sliding window dataset
SEQ_LEN = 16  # context window: 16 notes

class MelodyDataset(Dataset):
    def __init__(self, notes, seq_len, note_to_idx):
        self.seq_len = seq_len
        self.data = [note_to_idx[n] for n in notes]

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:idx + self.seq_len], dtype=torch.long)
        y = torch.tensor(self.data[idx + self.seq_len], dtype=torch.long)
        return x, y

dataset = MelodyDataset(all_notes, SEQ_LEN, note_to_idx)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Dataset size: {len(dataset)} windows")
print(f"Example input shape: {dataset[0][0].shape}")

## LSTM Model Architecture

Our model has three components:

1. **Embedding layer**: converts each note index into a dense vector (dimension 64).
   This is analogous to word embeddings in NLP — notes that function similarly
   (e.g., octave equivalents) may end up with similar embeddings.

2. **2-layer LSTM**: processes the embedded sequence step by step, maintaining a
   hidden state that accumulates context. Two stacked LSTM layers give the model
   more capacity to learn hierarchical patterns.

3. **Linear output layer**: maps the final hidden state to a probability distribution
   over the vocabulary (one logit per possible next note).

For generation, we use **temperature sampling**:
- Divide logits by temperature T before applying softmax
- T < 1.0: sharper distribution, more repetitive/predictable output
- T = 1.0: sample from the learned distribution
- T > 1.0: flatter distribution, more random/surprising output

In [ ]:
class LSTMMelodyModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_size=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        # x: (batch, seq_len)
        emb = self.embedding(x)           # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(emb, hidden)  # (batch, seq_len, hidden_size)
        out = self.dropout(out[:, -1, :])     # take last time step
        logits = self.fc(out)                  # (batch, vocab_size)
        return logits, hidden

    def generate(self, seed_seq, length=64, temperature=1.0):
        """Generate a melody given a seed sequence."""
        self.eval()
        generated = list(seed_seq)
        current = torch.tensor([seed_seq], dtype=torch.long).to(device)
        hidden = None

        with torch.no_grad():
            for _ in range(length):
                logits, hidden = self.forward(current, hidden)
                # Apply temperature
                logits = logits / temperature
                probs = torch.softmax(logits, dim=-1)
                # Sample from distribution
                next_idx = torch.multinomial(probs, 1).item()
                generated.append(next_idx)
                # Next input is just the predicted token
                current = torch.tensor([[next_idx]], dtype=torch.long).to(device)

        self.train()
        return generated

model = LSTMMelodyModel(vocab_size).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

We train with cross-entropy loss: the model learns to assign high probability
to the correct next note. With our small dataset, training should converge
quickly (under a minute on GPU, a few minutes on CPU).

In [ ]:
# Training configuration
NUM_EPOCHS = 100
LEARNING_RATE = 0.002

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

# Training loop
losses = []
model.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    n_batches = 0

    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits, _ = model(x_batch)
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    scheduler.step()
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses, linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('LSTM Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal loss: {losses[-1]:.4f}")

## Generation

Now we generate melodies at three different temperatures to hear the effect:

- **Temperature 0.5**: Conservative — the model sticks close to learned patterns,
  producing repetitive but coherent output.
- **Temperature 1.0**: Balanced — samples from the learned distribution as-is.
- **Temperature 1.5**: Adventurous — more random choices, potentially surprising
  but also more likely to produce incoherent sequences.

We use the first few notes of "Twinkle Twinkle" as a seed.

In [ ]:
def indices_to_midi(indices, idx_to_note, filename, note_duration=0.25, velocity=100):
    """Convert a sequence of note indices to a MIDI file."""
    midi = pretty_midi.PrettyMIDI(initial_tempo=120)
    piano = pretty_midi.Instrument(program=0, name='Piano')

    time = 0.0
    for idx in indices:
        note_val = idx_to_note[idx]
        if note_val != REST:  # skip rests
            note = pretty_midi.Note(
                velocity=velocity,
                pitch=note_val,
                start=time,
                end=time + note_duration * 0.9  # slight gap between notes
            )
            piano.notes.append(note)
        time += note_duration

    midi.instruments.append(piano)
    midi.write(filename)
    return midi


def plot_piano_roll(indices, idx_to_note, title="Piano Roll"):
    """Simple piano roll visualization."""
    notes = [idx_to_note[i] for i in indices]
    times = range(len(notes))

    fig, ax = plt.subplots(figsize=(14, 4))
    for t, n in zip(times, notes):
        if n != REST:
            ax.barh(n, 0.8, left=t, height=0.8, color='steelblue', alpha=0.7)

    ax.set_xlabel('Time Step')
    ax.set_ylabel('MIDI Pitch')
    ax.set_title(title)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()


# Seed sequence: first 8 notes of Twinkle
seed_notes = [60, 60, 67, 67, 69, 69, 67, REST]
seed_indices = [note_to_idx[n] for n in seed_notes]

# Extend seed to SEQ_LEN if needed
while len(seed_indices) < SEQ_LEN:
    seed_indices = [note_to_idx[REST]] + seed_indices

seed_indices = seed_indices[-SEQ_LEN:]  # take last SEQ_LEN

temperatures = [0.5, 1.0, 1.5]
generated_sequences = {}

os.makedirs('output', exist_ok=True)

for temp in temperatures:
    gen = model.generate(seed_indices, length=64, temperature=temp)
    generated_sequences[temp] = gen

    filename = f'output/lstm_temp_{temp}.mid'
    indices_to_midi(gen, idx_to_note, filename)
    print(f"Temperature {temp}: saved to {filename}")

    # Show piano roll
    plot_piano_roll(gen, idx_to_note, title=f'Generated Melody (Temperature = {temp})')

## Listening and Analysis

Let us synthesize the generated MIDI files to audio and listen to the results.
Pay attention to:

- **Repetition**: Does the melody get stuck in loops? (common at low temperature)
- **Coherence**: Do the note sequences sound like plausible melodies?
- **Range**: Does the model stay within a reasonable pitch range?
- **Comparison to Markov**: How does this compare to Week 3's Markov chains?

In [ ]:
# Synthesize MIDI to audio using pretty_midi's built-in FluidSynth
for temp in temperatures:
    filename = f'output/lstm_temp_{temp}.mid'
    midi = pretty_midi.PrettyMIDI(filename)

    # Synthesize to audio
    try:
        audio = midi.fluidsynth(fs=22050)
        print(f"\nTemperature {temp}:")
        display(Audio(audio, rate=22050))
    except Exception as e:
        print(f"FluidSynth not available ({e}). Listen to the MIDI files directly.")
        print(f"  -> {filename}")

# Pitch distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, temp in zip(axes, temperatures):
    gen = generated_sequences[temp]
    pitches = [idx_to_note[i] for i in gen if idx_to_note[i] != REST]
    ax.hist(pitches, bins=range(min(pitches), max(pitches)+2), color='steelblue', alpha=0.7)
    ax.set_title(f'Temp = {temp}')
    ax.set_xlabel('MIDI Pitch')
    if ax == axes[0]:
        ax.set_ylabel('Count')

fig.suptitle('Pitch Distributions at Different Temperatures', fontsize=13)
plt.tight_layout()
plt.show()

# Interval distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, temp in zip(axes, temperatures):
    gen = generated_sequences[temp]
    pitches = [idx_to_note[i] for i in gen if idx_to_note[i] != REST]
    intervals = [pitches[i+1] - pitches[i] for i in range(len(pitches)-1)]
    ax.hist(intervals, bins=range(min(intervals)-1, max(intervals)+2), color='coral', alpha=0.7)
    ax.set_title(f'Temp = {temp}')
    ax.set_xlabel('Interval (semitones)')
    if ax == axes[0]:
        ax.set_ylabel('Count')

fig.suptitle('Interval Distributions at Different Temperatures', fontsize=13)
plt.tight_layout()
plt.show()

print("\n--- Key Observations ---")
print("Low temperature (0.5): More repetitive, stays close to training patterns")
print("Medium temperature (1.0): Balanced between learned patterns and novelty")
print("High temperature (1.5): More varied but potentially less coherent")
print("\nCompare these to your Markov chain output from Week 3!")